We're building a custom dataset by combining ETF options chains from yfinance with earnings dates for each ETF's top holdings, scraped from stockanalysis.com. We verified scraping is allowed via their robots.txt. Our main question is whether implied volatility in ETF options responds to earnings events from the ETF's largest individual holdings

# Section 1 — Polite & Robust Scraping Compliance Check

In [ ]:
# Imports

import time
import logging
import requests
from urllib.robotparser import RobotFileParser
from urllib.parse import urljoin

In [ ]:
# Shared scraping configuration

# Single source of truth for headers and delay used across
# all scraping sections of this notebook.

BASE_URL   = "https://stockanalysis.com"
ROBOTS_URL = urljoin(BASE_URL, "/robots.txt")
DELAY      = 1.5   # seconds between consecutive requests

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger(__name__)

print(f"User-Agent : {HEADERS['User-Agent']}")
print(f"Delay      : {DELAY}s between requests")
print(f"Robots URL : {ROBOTS_URL}")

In [ ]:
# Fetch and display robots.txt

try:
    resp = requests.get(ROBOTS_URL, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    robots_text = resp.text
    print(f"✓ robots.txt fetched ({len(robots_text)} chars)\n")
    print("=" * 50)
    print(robots_text)
    print("=" * 50)
except requests.RequestException as e:
    robots_text = ""
    print(f"⚠ Could not fetch robots.txt: {e}")
    print("Proceeding with caution — assuming no restrictions.")

In [ ]:
# Parse robots.txt and check our URLs

URLS_TO_CHECK = [
    "/etf/qqq/holdings/",
    "/etf/voo/holdings/",
    "/etf/arkq/holdings/",
    "/etf/botz/holdings/",
    "/stocks/aapl/statistics/",
    "/stocks/nvda/statistics/",
]

# ── Detect blank Disallow (means "allow everything") ────────
# Python's RobotFileParser misinterprets a blank Disallow: line
# as "block all" when it actually means "block nothing."
# We check for this manually before falling back to the parser.

blank_disallow = (
    "User-agent: *\nDisallow:\n" in robots_text
    or "User-agent: *\nDisallow: \n" in robots_text
)

if blank_disallow:
    # Blank Disallow = no restrictions for all crawlers
    print("✓ robots.txt has blank Disallow for User-agent: *")
    print("  This means ALL paths are permitted — no restrictions.\n")

    print(f"{'URL':<45} {'Allowed?'}")
    print("-" * 55)
    for path in URLS_TO_CHECK:
        print(f"{path:<45} ✓ Allowed")

    print()
    print("✓ All scraped URLs are permitted by robots.txt")

else:
    # Fall back to RobotFileParser for normal robots.txt rules
    rp = RobotFileParser()
    rp.set_url(ROBOTS_URL)

    try:
        rp.read()
        print("✓ robots.txt parsed successfully\n")
    except Exception as e:
        print(f"⚠ Could not parse robots.txt: {e}\n")

    print(f"{'URL':<45} {'Allowed?'}")
    print("-" * 55)

    all_allowed = True
    for path in URLS_TO_CHECK:
        full_url = urljoin(BASE_URL, path)
        allowed  = rp.can_fetch("*", full_url)
        status   = "✓ Allowed" if allowed else "✗ DISALLOWED"
        if not allowed:
            all_allowed = False
        print(f"{path:<45} {status}")

    print()
    if all_allowed:
        print("✓ All scraped URLs are permitted by robots.txt")
    else:
        print("⚠ Some URLs are disallowed — review before scraping")

In [ ]:
# Demonstrate robust request wrapper

def polite_get(url: str, delay: float = DELAY) -> requests.Response | None:
    """
    Perform a single GET request with:
      - Descriptive User-Agent header
      - Timeout to avoid hanging indefinitely
      - try/except for all HTTP and connection errors
      - Logged warning on failure instead of crash
      - Polite delay after every request

    Parameters
    ----------
    url   : Full URL to fetch.
    delay : Seconds to sleep after the request (default: DELAY).

    Returns
    -------
    requests.Response if successful, None otherwise.
    """
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()   # raises for 4xx / 5xx
        return resp

    except requests.exceptions.Timeout:
        log.warning("Timeout fetching %s — skipping.", url)
    except requests.exceptions.ConnectionError:
        log.warning("Connection error fetching %s — skipping.", url)
    except requests.exceptions.HTTPError as e:
        status = e.response.status_code
        if status == 404:
            log.warning("404 Not Found: %s — page does not exist, skipping.", url)
        elif status == 429:
            log.warning("429 Too Many Requests: %s — slow down.", url)
        elif 400 <= status < 500:
            log.warning("Client error %d for %s — skipping.", status, url)
        elif 500 <= status < 600:
            log.warning("Server error %d for %s — skipping.", status, url)
    except requests.exceptions.RequestException as e:
        log.warning("Unexpected request error for %s: %s — skipping.", url, e)
    finally:
        time.sleep(delay)   # always delay, even on failure

    return None


In [ ]:
# Live test of polite_get on known URLs

test_urls = [
    "https://stockanalysis.com/etf/qqq/holdings/",   # should succeed
    "https://stockanalysis.com/stocks/aapl/statistics/",  # should succeed
    "https://stockanalysis.com/stocks/tlv:%20eslt/statistics/",  # 404 — known bad
]

print("Testing polite_get on sample URLs:\n")
for url in test_urls:
    print(f"→ {url}")
    resp = polite_get(url)
    if resp:
        print(f"  ✓ Status {resp.status_code} — {len(resp.text):,} chars received\n")
    else:
        print(f"  ✗ Skipped gracefully\n")

In [ ]:
# Compliance summary

print("=" * 55)
print("POLITE SCRAPING COMPLIANCE SUMMARY")
print("=" * 55)
print(f"  User-Agent header    : ✓ Set on all requests")
print(f"  Request delay        : ✓ {DELAY}s between each request")
print(f"  Error handling       : ✓ try/except for 4xx, 5xx,")
print(f"                             timeouts, connection errors")
print(f"  Failure behavior     : ✓ Logs warning, skips, no crash")
print(f"  robots.txt checked   : ✓ All scraped URLs permitted")
print(f"  Pages scraped        : /etf/*/holdings/")
print(f"                         /stocks/*/statistics/")
print("=" * 55)

# Section 1 — Price History

In [ ]:
# Imports

import pandas as pd
from data_collection import get_price_history

In [ ]:
# Configuration

TICKERS = ["VOO", "QQQ", "ARKQ", "BOTZ"]
PERIOD  = "6mo"
OUTPUT_CSV = "price_history.csv"

In [ ]:
# Fetch price history for all tickers

frames = []

for ticker in TICKERS:
    print(f"Fetching price history: {ticker} ...", end=" ")
    try:
        df = get_price_history(ticker, period=PERIOD)
        frames.append(df)
        print(f"✓  {len(df):,} rows")
    except ValueError as e:
        print(f"✗  Skipped — {e}")

if not frames:
    raise RuntimeError("No data collected. Check ticker symbols and network.")

prices = pd.concat(frames, ignore_index=False)

In [ ]:
# Clean and standardize

# Reset index so Date becomes a regular column
prices = prices.reset_index()

# Rename to match acceptance criteria exactly
prices = prices.rename(columns={"index": "Date", "ticker": "ticker"})

# Keep only required columns
KEEP_COLS = ["ticker", "Date", "Open", "High", "Low", "Close", "Volume"]
cols = [c for c in KEEP_COLS if c in prices.columns]
prices = prices[cols]

# Ensure correct dtypes
prices["Date"]   = pd.to_datetime(prices["Date"]).dt.tz_localize(None)
prices["Open"]   = pd.to_numeric(prices["Open"],   errors="coerce")
prices["High"]   = pd.to_numeric(prices["High"],   errors="coerce")
prices["Low"]    = pd.to_numeric(prices["Low"],    errors="coerce")
prices["Close"]  = pd.to_numeric(prices["Close"],  errors="coerce")
prices["Volume"] = pd.to_numeric(prices["Volume"], errors="coerce")

# Sort for readability
prices = prices.sort_values(["ticker", "Date"]).reset_index(drop=True)

prices.head(10)


In [ ]:
# Verify no major gaps in the time series

print("=== Gap Analysis ===\n")

for ticker in TICKERS:
    subset = prices[prices["ticker"] == ticker].copy()

    if subset.empty:
        print(f"{ticker}: No data.\n")
        continue

    # Expected trading days (roughly 5 per week, minus holidays)
    date_range = pd.bdate_range(subset["Date"].min(), subset["Date"].max())
    expected   = len(date_range)
    actual     = len(subset)
    missing    = expected - actual

    # Find specific gap dates
    actual_dates   = set(subset["Date"].dt.date)
    expected_dates = set(date_range.date)
    gap_dates      = sorted(expected_dates - actual_dates)

    print(f"{ticker}:")
    print(f"  Date range : {subset['Date'].min().date()} → {subset['Date'].max().date()}")
    print(f"  Expected   : {expected} trading days")
    print(f"  Actual     : {actual} rows")
    print(f"  Missing    : {missing} days", end="")

    if missing == 0:
        print(" ✓ No gaps")
    elif missing <= 5:
        print(f" — likely holidays: {gap_dates}")
    else:
        print(f" ⚠ Potential data issue — gap dates: {gap_dates[:10]}")
    print()

In [ ]:
# Quick summary stats

summary = (
    prices.groupby("ticker")
    .agg(
        rows        = ("Close", "count"),
        start_date  = ("Date",  "min"),
        end_date    = ("Date",  "max"),
        avg_close   = ("Close", "mean"),
        min_close   = ("Close", "min"),
        max_close   = ("Close", "max"),
        avg_volume  = ("Volume","mean"),
    )
    .round(2)
)
print(summary)


In [ ]:
# Save to CSV

prices.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved CSV → {OUTPUT_CSV}  ({prices.shape[0]:,} rows, {prices.shape[1]} columns)")


In [ ]:
# Reload check

reloaded = pd.read_csv(OUTPUT_CSV, parse_dates=["Date"])
assert reloaded.shape == prices.shape, "Reload shape mismatch!"
print(f"Reload check passed — {reloaded.shape[0]:,} rows, {reloaded.shape[1]} columns.")
reloaded.head()

# Section 2 — Options Chain

In [ ]:
# Imports 

import pandas as pd
from data_collection import get_options_chain

In [ ]:
# Configuration

TICKERS = ["VOO", "QQQ", "ARKQ", "BOTZ"]

KEEP_COLS = [
    "ticker",
    "expiration",
    "contractType",   # renamed from option_type for clarity
    "strike",
    "bid",
    "ask",
    "volume",
    "openInterest",
    "impliedVolatility",
]

OUTPUT_CSV     = "options_chain.csv"
OUTPUT_PARQUET = "options_chain.parquet"

In [ ]:
# Fetch options chain for all tickers

frames = []

for ticker in TICKERS:
    print(f"Fetching options chain: {ticker} ...", end=" ")
    try:
        df = get_options_chain(ticker)

        # Rename option_type -> contractType to match acceptance criteria
        df = df.rename(columns={"option_type": "contractType"})

        # Keep only the required columns (drop any that don't exist gracefully)
        cols = [c for c in KEEP_COLS if c in df.columns]
        df = df[cols]

        frames.append(df)
        print(f"✓  {len(df):,} rows")

    except ValueError as e:
        print(f"✗  Skipped — {e}")

if not frames:
    raise RuntimeError("No data collected. Check ticker symbols and network.")

options = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows collected: {len(options):,}")

In [ ]:
# Light cleaning

# Ensure correct dtypes
options["expiration"]       = pd.to_datetime(options["expiration"])
options["strike"]           = pd.to_numeric(options["strike"],           errors="coerce")
options["bid"]              = pd.to_numeric(options["bid"],              errors="coerce")
options["ask"]              = pd.to_numeric(options["ask"],              errors="coerce")
options["volume"]           = pd.to_numeric(options["volume"],           errors="coerce")
options["openInterest"]     = pd.to_numeric(options["openInterest"],     errors="coerce")
options["impliedVolatility"]= pd.to_numeric(options["impliedVolatility"],errors="coerce")

# Drop rows where strike or IV is missing — not useful for analysis
before = len(options)
options = options.dropna(subset=["strike", "impliedVolatility"])
dropped = before - len(options)
if dropped:
    print(f"Dropped {dropped:,} rows with missing strike or impliedVolatility.")

# Sort for readability
options = options.sort_values(
    ["ticker", "expiration", "contractType", "strike"]
).reset_index(drop=True)

options.head(10)


In [ ]:
# Quick summary

summary = (
    options.groupby(["ticker", "contractType"])
    .agg(
        expirations   = ("expiration", "nunique"),
        contracts     = ("strike",     "count"),
        avg_iv        = ("impliedVolatility", "mean"),
        total_oi      = ("openInterest", "sum"),
    )
    .round(4)
)
print(summary)

In [ ]:
# Save to disk

options.to_csv(OUTPUT_CSV, index=False)
print(f"Saved CSV     → {OUTPUT_CSV}  ({options.shape[0]:,} rows)")

# Section 3 — Spot Price Join

In [ ]:
# Load saved data

options = pd.read_csv("options_chain.csv", parse_dates=["expiration"])
prices  = pd.read_csv("price_history.csv", parse_dates=["Date"])

print(f"Options rows : {len(options):,}")
print(f"Price rows   : {len(prices):,}")

In [ ]:
# Build spot price lookup table

# We want the most recent closing price for each ticker as of today (the snapshot date when data was collected).
# Since price history is sorted oldest -> newest, the last row per ticker is the most recent closing price.

spot = (
    prices.sort_values("Date")
    .groupby("ticker", as_index=False)
    .last()                          # most recent row per ticker
    [["ticker", "Date", "Close"]]
    .rename(columns={
        "Close": "spot_price",
        "Date" : "spot_date",
    })
)

print("\nSpot prices used for join:")
print(spot.to_string(index=False))

In [ ]:
# Join spot price onto options chain

# Simple left join on ticker — every option row gets the spot price of its underlying ETF on the collection date.

options_with_spot = options.merge(spot[["ticker", "spot_price"]], 
                                   on="ticker", 
                                   how="left")

print(f"\nRows before join : {len(options):,}")
print(f"Rows after join  : {len(options_with_spot):,}")

In [ ]:
# Verify no unexpected NaNs in spot_price

nan_count = options_with_spot["spot_price"].isna().sum()

if nan_count == 0:
    print("✓ No NaNs in spot_price — join successful.")
else:
    # Show which tickers failed to join
    problem_tickers = (
        options_with_spot[options_with_spot["spot_price"].isna()]
        ["ticker"]
        .unique()
    )
    print(f"⚠ {nan_count:,} NaNs found in spot_price.")
    print(f"  Affected tickers: {problem_tickers}")
    print("  Check that these tickers exist in price_history.csv.")


In [ ]:
# Compute moneyness

# Moneyness = strike / spot price
# = 1.0  → at the money  (strike == spot)
# > 1.0  → out of the money for calls / in the money for puts
# < 1.0  → in the money for calls  / out of the money for puts

options_with_spot["moneyness"] = (
    options_with_spot["strike"] / options_with_spot["spot_price"]
).round(4)

print("\nMoneyness sample:")
print(
    options_with_spot[["ticker", "expiration", "contractType", 
                        "strike", "spot_price", "moneyness"]]
    .head(10)
    .to_string(index=False)
)

In [ ]:
# Quick sanity check

summary = (
    options_with_spot.groupby("ticker")
    .agg(
        contracts  = ("strike",     "count"),
        spot_price = ("spot_price", "first"),
        min_strike = ("strike",     "min"),
        max_strike = ("strike",     "max"),
        min_money  = ("moneyness",  "min"),
        max_money  = ("moneyness",  "max"),
    )
    .round(4)
)
print("\nSummary by ticker:")
print(summary)

In [ ]:
# Save merged dataset

OUTPUT_MERGED = "options_with_spot.csv"

options_with_spot.to_csv(OUTPUT_MERGED, index=False)
print(f"\nSaved → {OUTPUT_MERGED}  ({options_with_spot.shape[0]:,} rows, "
      f"{options_with_spot.shape[1]} columns)")


In [ ]:
# Reload check

reloaded = pd.read_csv(OUTPUT_MERGED, parse_dates=["expiration"])
assert reloaded.shape == options_with_spot.shape, "Reload shape mismatch!"
print(f"Reload check passed — {reloaded.shape[0]:,} rows, "
      f"{reloaded.shape[1]} columns.")
reloaded.head()

# Section 4 — Holdings

In [ ]:
# Imports

import time
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [ ]:
# Configuration

TICKERS     = ["VOO", "QQQ", "ARKQ", "BOTZ"]
OUTPUT_CSV  = "etf_holdings.csv"
DELAY       = 1.5   # seconds between requests — polite scraping

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

def build_url(ticker: str) -> str:
    return f"https://stockanalysis.com/etf/{ticker.lower()}/holdings/"


In [ ]:
# Scraper function

def scrape_holdings(ticker: str) -> pd.DataFrame:
    """
    Scrape top holdings from stockanalysis.com for a given ETF ticker.

    Returns a DataFrame with columns:
        holding_ticker, company_name, weight_pct, etf_ticker

    Raises ValueError if the page or table cannot be parsed.
    """
    url = build_url(ticker)

    # --- Fetch page ---
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
    except requests.RequestException as e:
        raise ValueError(f"HTTP request failed for {ticker}: {e}") from e

    soup = BeautifulSoup(resp.text, "lxml")

    # --- Find the holdings table ---
    table = soup.find("table")
    if table is None:
        raise ValueError(
            f"No table found for {ticker}. "
            "Page structure may have changed."
        )

    # --- Parse rows ---
    rows = []
    for tr in table.find("tbody").find_all("tr"):
        cells = tr.find_all("td")
        if len(cells) < 4:
            continue

        # Column order: No. | Symbol | Name | % Weight | Shares
        symbol  = cells[1].get_text(strip=True)
        name    = cells[2].get_text(strip=True)
        weight  = cells[3].get_text(strip=True)

        # Clean weight — strip "%" and convert to float
        try:
            weight_float = float(weight.replace("%", "").strip())
        except ValueError:
            weight_float = float("nan")

        rows.append({
            "holding_ticker": symbol,
            "company_name"  : name,
            "weight_pct"    : weight_float,
            "etf_ticker"    : ticker.upper(),
        })

    if not rows:
        raise ValueError(f"Table found but no rows parsed for {ticker}.")

    return pd.DataFrame(rows)


In [ ]:
# Fetch holdings for all tickers

frames = []

for i, ticker in enumerate(TICKERS):
    print(f"Scraping holdings: {ticker} ...", end=" ")
    try:
        df = scrape_holdings(ticker)
        frames.append(df)
        print(f"✓  {len(df)} holdings")
    except ValueError as e:
        print(f"✗  Skipped — {e}")

    # Polite delay between requests (skip after last ticker)
    if i < len(TICKERS) - 1:
        time.sleep(DELAY)

if not frames:
    raise RuntimeError("No holdings data collected.")

holdings = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows: {len(holdings):,}")

In [ ]:
# Verify and preview

# Check for unexpected NaNs
nan_weights = holdings["weight_pct"].isna().sum()
if nan_weights == 0:
    print("✓ No NaNs in weight_pct")
else:
    print(f"⚠ {nan_weights} NaN weights — inspect rows below:")
    print(holdings[holdings["weight_pct"].isna()])

# Coverage check — confirm all 4 ETFs scraped
scraped = holdings["etf_ticker"].unique()
missing = set(TICKERS) - set(scraped)
if missing:
    print(f"⚠ Missing ETFs: {missing}")
else:
    print(f"✓ All ETFs present: {sorted(scraped)}")

# Preview top 5 holdings per ETF
print("\nTop 5 holdings per ETF:")
print(
    holdings.groupby("etf_ticker")
    .head(5)
    .to_string(index=False)
)

In [ ]:
# Summary stats

summary = (
    holdings.groupby("etf_ticker")
    .agg(
        num_holdings    = ("holding_ticker", "count"),
        top10_weight    = ("weight_pct",     lambda x: x.head(10).sum()),
        max_weight      = ("weight_pct",     "max"),
    )
    .round(2)
)
print(summary)

In [ ]:
# Save to CSV

holdings.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved → {OUTPUT_CSV}  ({holdings.shape[0]:,} rows, "
      f"{holdings.shape[1]} columns)")

In [ ]:
# Reload check

reloaded = pd.read_csv(OUTPUT_CSV)
assert reloaded.shape == holdings.shape, "Reload shape mismatch!"
print(f"Reload check passed — {reloaded.shape[0]:,} rows.")
reloaded.head()

# Section 5 — Earnings Dates

In [ ]:
# Imports

import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [ ]:
# Configuration

OUTPUT_CSV = "earnings_dates.csv"
DELAY      = 1.5   # seconds between requests — polite scraping

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

def build_url(ticker: str) -> str:
    return f"https://stockanalysis.com/stocks/{ticker.lower()}/statistics/"

In [ ]:
# Load holdings from previous section

holdings = pd.read_csv("etf_holdings.csv")

print(f"Loaded {len(holdings):,} holdings across "
      f"{holdings['etf_ticker'].nunique()} ETFs")

# Deduplicate — same stock can appear in multiple ETFs (e.g. AAPL in VOO + QQQ)
# We only need to scrape each holding_ticker once
unique_holdings = (
    holdings[["holding_ticker", "etf_ticker"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

unique_tickers = unique_holdings["holding_ticker"].unique()
print(f"Unique holding tickers to scrape: {len(unique_tickers)}")


In [ ]:
# Scraper function

def scrape_earnings_date(ticker: str) -> str | None:
    """
    Scrape the next estimated earnings date for a stock from
    stockanalysis.com/stocks/{ticker}/statistics/.

    Returns the earnings date as a string (e.g. 'Apr 30, 2026'),
    or None if not found or the request fails.
    """
    url = build_url(ticker)

    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"  ⚠ HTTP error for {ticker}: {e}")
        return None

    soup = BeautifulSoup(resp.text, "lxml")

    # Find the "Important Dates" section — look for a table row
    # where the first cell says "Earnings Date"
    for row in soup.find_all("tr"):
        cells = row.find_all("td")
        if len(cells) == 2 and "Earnings Date" in cells[0].get_text():
            return cells[1].get_text(strip=True)

    return None   # No earnings date found

In [ ]:
# Scrape earnings dates for all unique holdings

earnings_map = {}   # ticker -> earnings_date string or None

for i, ticker in enumerate(unique_tickers):
    date = scrape_earnings_date(ticker)
    earnings_map[ticker] = date

    status = f"✓  {date}" if date else "—  No date found"
    print(f"[{i+1:>3}/{len(unique_tickers)}] {ticker:<8} {status}")

    # Polite delay between requests
    if i < len(unique_tickers) - 1:
        time.sleep(DELAY)

# lots of 404 errors // no date found

In [ ]:
# Build earnings DataFrame

# Join earnings dates back onto the full holdings table
# so each row has both etf_ticker and earnings_date

earnings_df = (
    unique_holdings
    .copy()
    .assign(earnings_date=lambda df: df["holding_ticker"].map(earnings_map))
)

# Parse to datetime where possible — keep NaT for missing dates
earnings_df["earnings_date"] = pd.to_datetime(
    earnings_df["earnings_date"], errors="coerce"
)

print(f"\nTotal rows       : {len(earnings_df):,}")
print(f"With date        : {earnings_df['earnings_date'].notna().sum():,}")
print(f"Missing date     : {earnings_df['earnings_date'].isna().sum():,}")


In [ ]:
# Preview results

print("\nUpcoming earnings (soonest first):")
print(
    earnings_df
    .dropna(subset=["earnings_date"])
    .sort_values("earnings_date")
    .head(20)
    .to_string(index=False)
)

print("\nHoldings with no earnings date:")
print(
    earnings_df[earnings_df["earnings_date"].isna()]
    .to_string(index=False)
)

In [ ]:
# Save to CSV

earnings_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved → {OUTPUT_CSV}  ({earnings_df.shape[0]:,} rows, "
      f"{earnings_df.shape[1]} columns)")
